In [1]:
import time
import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import LabelEncoder

train_raw = pd.read_csv('C:/Users/user/dacon/Smart_Logistics/data/train.csv')
test_raw  = pd.read_csv('C:/Users/user/dacon/Smart_Logistics/data/test.csv')
layout    = pd.read_csv('C:/Users/user/dacon/Smart_Logistics/data/layout_info.csv')

layout_clean = layout[['layout_id', 'layout_type', 'aisle_width_avg',
                        'intersection_count', 'one_way_ratio', 'pack_station_count',
                        'charger_count', 'layout_compactness', 'zone_dispersion',
                        'robot_total', 'floor_area_sqm', 'ceiling_height_m',
                        'fire_sprinkler_count', 'emergency_exit_count']]

train = train_raw.merge(layout_clean, on='layout_id', how='left')
test  = test_raw.merge(layout_clean, on='layout_id', how='left')

le = LabelEncoder()
train['layout_type'] = le.fit_transform(train['layout_type'].fillna('unknown'))
test['layout_type']  = le.transform(test['layout_type'].fillna('unknown'))

TARGET   = 'avg_delay_minutes_next_30m'
ID_COLS  = ['ID', 'layout_id', 'scenario_id']
feature_cols = [c for c in train.columns if c not in ID_COLS + [TARGET]]

# CatBoost v15 예측값
cat_preds = np.zeros(len(test))
for fold in range(1, 6):
    m = joblib.load(f'C:/Users/user/dacon/Smart_Logistics/models/cat_fold{fold}.pkl')
    cat_preds += m.predict(test[feature_cols]) / 5
    print(f'CatBoost Fold {fold} done')

# LightGBM v14 예측값
lgbm_preds = np.zeros(len(test))
for fold in range(1, 6):
    m = joblib.load(f'C:/Users/user/dacon/Smart_Logistics/models/lgbm_optuna_lr03_fold{fold}.pkl')
    lgbm_preds += m.predict(test[feature_cols]) / 5
    print(f'LightGBM Fold {fold} done')

# 가중 앙상블 3가지 버전 계산
test_id = pd.read_csv('C:/Users/user/dacon/Smart_Logistics/data/test.csv')['ID']

for cat_w, lgbm_w in [(0.7, 0.3), (0.8, 0.2), (0.9, 0.1)]:
    preds = cat_w * cat_preds + lgbm_w * lgbm_preds
    fname = f'submission_v19_cat{int(cat_w*10)}_lgbm{int(lgbm_w*10)}_ensemble.csv'
    pd.DataFrame({
        'ID': test_id,
        'avg_delay_minutes_next_30m': preds
    }).to_csv(f'C:/Users/user/dacon/Smart_Logistics/results/{fname}', index=False)
    print(f'저장완료: {fname}')

CatBoost Fold 1 done
CatBoost Fold 2 done
CatBoost Fold 3 done
CatBoost Fold 4 done
CatBoost Fold 5 done
LightGBM Fold 1 done
LightGBM Fold 2 done
LightGBM Fold 3 done
LightGBM Fold 4 done
LightGBM Fold 5 done
저장완료: submission_v19_cat7_lgbm3_ensemble.csv
저장완료: submission_v19_cat8_lgbm2_ensemble.csv
저장완료: submission_v19_cat9_lgbm1_ensemble.csv
